# 201 · Dynamic vs IDL binary

Companion to [Dynamic vs IDL binary](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/dynamic-vs-idl-binary/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/dynamic_vs_idl_binary.ipynb)

**Goal:** compare MessagePack-class (dynamic) vs Protobuf-class (IDL) encodings on the same logical record.

**Why this lab:** once you reject text JSON for a hop, you still choose between flexible documents and a long-lived product contract.

**How to use:** size JSON vs MessagePack map/array vs an IDL field-number sketch; then add an ad-hoc field.

**Expect:** dynamic forms accept extra keys without codegen; IDL sketch has no slot until you allocate a new field number.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth. Compare within one language and paradigm—not global format rankings.


In [ ]:
import json
import struct

RECORD = {
    "order_id": 1001,
    "sku": "ABC-42",
    "qty": 3,
    "price_cents": 1999,
}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack recommended for this notebook")



In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_idl(r: dict) -> bytes:
    # 1 order_id, 2 sku, 3 qty, 4 price_cents — all varint/string
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(r["order_id"])
    b = r["sku"].encode()
    out += encode_key(2, 2) + encode_varint(len(b)) + b
    out += encode_key(3, 0) + encode_varint(r["qty"])
    out += encode_key(4, 0) + encode_varint(r["price_cents"])
    return bytes(out)


j = json.dumps(RECORD, separators=(",", ":")).encode()
idl = encode_idl(RECORD)
rows = [("JSON (text baseline)", len(j)), ("IDL binary sketch", len(idl))]
if HAS_MSGPACK:
    mp_map = msgpack.packb(RECORD, use_bin_type=True)
    mp_arr = msgpack.packb(
        [RECORD["order_id"], RECORD["sku"], RECORD["qty"], RECORD["price_cents"]],
        use_bin_type=True,
    )
    rows[1:1] = [
        ("MessagePack map", len(mp_map)),
        ("MessagePack array", len(mp_arr)),
    ]
print(f"{'encoding':24} {'bytes':>6}")
for name, n in rows:
    print(f"{name:24} {n:6}")
print("IDL hex:", idl.hex(" "))



## Decision frame (from the article)

**Why:** the durable axis is **flexibility vs contract investment**, not “which logo is popular.”

| Prefer dynamic binary when… | Prefer IDL binary when… |
|-----------------------------|-------------------------|
| Document shapes vary | Record is a multi-year product interface |
| You will not run an IDL toolchain | Multi-language stubs + field-number discipline |
| Validation at boundaries is enough | Compatibility rules must be explicit |

**Neither replaces JSON for public human-debuggable APIs without a plan.**

**Why it matters:** picking MessagePack “because binary” without validation, or Protobuf without process, both fail operationally.


## Flexibility demo

**Why:** dynamic maps absorb optional fields; IDL wires need an agreed number first.

**How:** add `note` to the record; pack with MessagePack/JSON; contrast with the fixed IDL sketch.

**Expect:** dynamic encodings grow to include `note`; the teaching IDL encoder still has only fields 1–4.

**Why it matters:** rapid product iteration favors dynamic models; stable multi-language RPCs favor IDL discipline.


In [ ]:
flexible = dict(RECORD)
flexible["note"] = "rush"
if HAS_MSGPACK:
    print("msgpack with extra field nbytes", len(msgpack.packb(flexible, use_bin_type=True)))
print("JSON with extra field:", json.dumps(flexible, separators=(",", ":")))
print("IDL sketch above has no slot for 'note' until you allocate field 5+ in the schema.")



## Takeaways

- Dynamic binary ≈ JSON data model, binary tags (often keys).
- IDL binary ≈ shared schema, field numbers, codegen.
- Choose by **contract investment**, not a single latency chart.

**Why it matters:** the wrong family creates either endless ad-hoc validation or brittle schema churn.

**Next:** [Compression vs format](./compression_vs_format.ipynb)
